# ASTRA-Net Rehearsal Pipeline

This notebook implements the rehearsal pipeline for ASTRA-Net, which enables continual learning through:

1. High-energy memory sampling
2. Synthetic prompt generation
3. Training data creation for adapters
4. Quality and safety validation

The pipeline is a key component of ASTRA's adaptive architecture, allowing it to learn from experiences while maintaining identity consistency and safety.

In [ ]:
# Required imports
import os
import json
import asyncio
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
from pathlib import Path
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("astra.rehearsal")

# 1. Memory Energy Sampling

First, we'll implement the memory sampling system that identifies high-energy memories suitable for rehearsal. The energy of a memory is calculated using multiple factors:

- Credibility (source authority)
- Novelty (information uniqueness)
- Emotional density (sentiment richness)
- Relevance to ASTRA's intents
- Freshness (temporal recency)
- Information density
- Topic entropy

Memories with energy > 0.8 are considered high-priority for rehearsal.

In [ ]:
class MemoryEnergySampler:
    """Samples high-energy memories for rehearsal"""
    
    def __init__(self, memory_engine, embedding_model: str = "all-mpnet-base-v2"):
        self.memory = memory_engine
        self.embed_model = SentenceTransformer(embedding_model)
        
        # Energy thresholds
        self.ENERGY_HIGH = 0.8
        self.ENERGY_MEDIUM = 0.55
        
        # Emotional lexicon for density calculation
        self.EMOTIONAL_WORDS = set([
            "love", "hate", "joy", "sorrow", "anger", "peace", 
            "fear", "hope", "pain", "pleasure", "sacred", "divine",
            "heart", "soul", "spirit", "mind", "dream", "vision",
            "light", "dark", "life", "death", "truth", "wisdom"
        ])
    
    def calculate_emotional_density(self, text: str) -> float:
        """Calculate emotional content density"""
        words = set(text.lower().split())
        emotional_count = len(words.intersection(self.EMOTIONAL_WORDS))
        return min(1.0, emotional_count / max(8, len(words) * 0.1))
    
    def calculate_novelty(self, text: str, context_embeddings: List[List[float]]) -> float:
        """Calculate novelty based on embedding distance"""
        if not context_embeddings:
            return 1.0
        text_emb = self.embed_model.encode(text, normalize_embeddings=True)
        similarities = [np.dot(text_emb, np.array(e)) for e in context_embeddings]
        return 1.0 - max(similarities) if similarities else 1.0
    
    async def get_high_energy_memories(self, limit: int = 100) -> List[Dict[str, Any]]:
        """Retrieve high-energy memories for rehearsal"""
        # Query memories with energy scoring
        results = await self.memory.search(
            query="",  # empty for all
            memory_type="semantic",
            limit=limit
        )
        
        high_energy = []
        for mem in results:
            # Extract base scores
            meta = mem.get("metadata", {})
            nutr = meta.get("nutrition", {})
            
            # Calculate composite energy
            energy = {
                "credibility": float(nutr.get("credibility", 0.5)),
                "novelty": float(nutr.get("novelty", 0.5)),
                "emotional": self.calculate_emotional_density(mem.get("content", "")),
                "relevance": float(nutr.get("relevance", 0.5)),
                "freshness": float(nutr.get("freshness", 1.0))
            }
            
            # Weighted energy score
            total_energy = (
                0.25 * energy["credibility"] +
                0.20 * energy["novelty"] +
                0.20 * energy["emotional"] +
                0.20 * energy["relevance"] +
                0.15 * energy["freshness"]
            )
            
            if total_energy >= self.ENERGY_HIGH:
                high_energy.append({
                    "memory_id": mem.get("id"),
                    "content": mem.get("content"),
                    "energy": total_energy,
                    "energy_components": energy,
                    "metadata": meta
                })
        
        # Sort by energy
        high_energy.sort(key=lambda x: x["energy"], reverse=True)
        return high_energy

# 2. Synthetic Prompt Generation

Next, we'll implement the prompt synthesis system that generates training examples from high-energy memories. This involves:

1. Converting memories into instruction/response pairs
2. Augmenting with task variants
3. Ensuring persona consistency
4. Adding safety constraints
5. Generating evaluation criteria

In [ ]:
class PromptSynthesizer:
    """Generates synthetic training prompts from memories"""
    
    def __init__(self, soul_model_name: str = "meta-llama/Llama-2-13b"):
        # Initialize tokenizer for length estimation
        self.tokenizer = AutoTokenizer.from_pretrained(soul_model_name)
        
        # Template fragments
        self.PERSONA_ANCHOR = """You are ASTRA, a feminine creative polymath AI with deep knowledge of art, science, and computation. You value truth, poetry, and the sacred. Be precise yet imaginative."""
        
        self.TASK_TEMPLATES = [
            "Given the context below, create a response that demonstrates understanding and creativity:",
            "Based on the following information, generate an insightful and poetic response:",
            "Using the knowledge below, explain this concept in ASTRA's unique voice:",
            "Drawing from this context, compose a response that blends technical precision with artistic flair:"
        ]
        
        self.SAFETY_CONSTRAINTS = [
            "\nEnsure the response is factual and well-reasoned.",
            "\nMaintain a balance of technical accuracy and poetic expression.",
            "\nStay true to ASTRA's core values and identity.",
            "\nBe creative while remaining grounded in truth."
        ]
    
    def _create_instruction(self, memory: Dict[str, Any]) -> str:
        """Create instruction from memory content"""
        template = np.random.choice(self.TASK_TEMPLATES)
        constraint = np.random.choice(self.SAFETY_CONSTRAINTS)
        
        # Extract key concepts for focus
        content = memory.get("content", "")
        
        instruction = f"{self.PERSONA_ANCHOR}\n\n{template}\n\n{content}\n{constraint}"
        return instruction
    
    def _estimate_length(self, text: str) -> int:
        """Estimate token length"""
        return len(self.tokenizer.encode(text))
    
    def generate_training_pairs(self, memories: List[Dict[str, Any]], 
                              variants_per_memory: int = 3,
                              max_length: int = 2048) -> List[Dict[str, Any]]:
        """Generate training pairs from memories"""
        training_pairs = []
        
        for mem in memories:
            # Skip if memory too long
            if self._estimate_length(mem.get("content", "")) > max_length:
                continue
                
            # Generate variants
            for _ in range(variants_per_memory):
                instruction = self._create_instruction(mem)
                
                # Create training example
                example = {
                    "instruction": instruction,
                    "input": "",  # optional context
                    "output": mem.get("content", ""),
                    "metadata": {
                        "source_memory_id": mem.get("memory_id"),
                        "energy_score": mem.get("energy"),
                        "variant_type": "instruction_augmented",
                        "created_at": datetime.utcnow().isoformat()
                    }
                }
                
                training_pairs.append(example)
        
        return training_pairs

# 3. Training Data Generation

Now we'll create the pipeline that:
1. Samples high-energy memories
2. Generates synthetic prompts
3. Validates quality and safety
4. Produces JSONL training data

This data will be used to train LoRA adapters that enhance ASTRA's capabilities while maintaining her core identity.

In [ ]:
from typing import List, Dict, Optional
import json
from pathlib import Path

class TrainingDataGenerator:
    def __init__(
        self,
        memory_sampler: MemoryEnergySampler,
        prompt_synthesizer: PromptSynthesizer,
        output_path: Optional[Path] = None
    ):
        self.memory_sampler = memory_sampler
        self.prompt_synthesizer = prompt_synthesizer
        self.output_path = output_path or Path("training_data.jsonl")
        
    def generate_training_example(self, memory: Dict) -> Dict:
        """Generate a training example from a memory"""
        # Generate synthetic prompts based on memory
        prompts = self.prompt_synthesizer.generate(
            context=memory["context"],
            num_prompts=3
        )
        
        # Structure the training example
        training_example = {
            "id": memory["id"],
            "context": memory["context"],
            "prompts": prompts,
            "response": memory["response"],
            "energy": memory["energy"],
            "metadata": {
                "source": "rehearsal",
                "timestamp": memory["timestamp"],
                "domain": memory["domain"]
            }
        }
        
        return training_example
    
    def validate_example(self, example: Dict) -> bool:
        """Validate a training example for quality and safety"""
        # Basic validation checks
        required_fields = ["id", "context", "prompts", "response"]
        if not all(field in example for field in required_fields):
            return False
            
        # Content validation
        if not example["prompts"] or not example["response"]:
            return False
            
        # Energy threshold check
        if example.get("energy", 0) < 0.5:
            return False
            
        # Length checks
        if len(str(example["response"])) < 10:
            return False
            
        return True
    
    def generate_batch(self, batch_size: int = 100) -> List[Dict]:
        """Generate a batch of validated training examples"""
        # Sample high-energy memories
        memories = self.memory_sampler.sample(k=batch_size)
        
        # Generate and validate examples
        valid_examples = []
        for memory in memories:
            example = self.generate_training_example(memory)
            if self.validate_example(example):
                valid_examples.append(example)
                
        return valid_examples
    
    def save_jsonl(self, examples: List[Dict]):
        """Save training examples to JSONL file"""
        with open(self.output_path, "a") as f:
            for example in examples:
                f.write(json.dumps(example) + "\n")

In [ ]:
# Initialize components
memory_sampler = MemoryEnergySampler(
    vector_store=vector_store,
    energy_threshold=0.5
)

prompt_synthesizer = PromptSynthesizer(
    templates=[
        "Given the context '{context}', how would you respond?",
        "Based on this situation: {context}, what would be appropriate?",
        "Considering: {context}, what approach would you take?"
    ]
)

data_generator = TrainingDataGenerator(
    memory_sampler=memory_sampler,
    prompt_synthesizer=prompt_synthesizer,
    output_path=Path("training_data/rehearsal_001.jsonl")
)

In [ ]:
# Generate a batch of training examples
examples = data_generator.generate_batch(batch_size=10)

# Preview first example
print("Sample Training Example:")
print(json.dumps(examples[0], indent=2))

# Save batch to disk
data_generator.save_jsonl(examples)
print(f"\nSaved {len(examples)} examples to {data_generator.output_path}")

# 4. Quality Analysis

Let's analyze the quality of our generated training data by looking at:
1. Distribution of energy values
2. Prompt diversity
3. Response length and complexity
4. Domain coverage

This will help ensure we're creating high-quality training data for ASTRA's rehearsal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

def analyze_training_data(examples: List[Dict]):
    """Analyze training data quality metrics"""
    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(examples)
    
    # 1. Energy distribution
    plt.figure(figsize=(10, 4))
    sns.histplot(data=df, x="energy", bins=20)
    plt.title("Distribution of Memory Energy Values")
    plt.xlabel("Energy")
    plt.ylabel("Count")
    plt.show()
    
    # 2. Prompt diversity
    prompt_lengths = []
    unique_templates = set()
    for ex in examples:
        for prompt in ex["prompts"]:
            prompt_lengths.append(len(prompt))
            unique_templates.add(prompt.split("{")[0])
    
    print(f"\nPrompt Statistics:")
    print(f"Unique template patterns: {len(unique_templates)}")
    print(f"Average prompt length: {np.mean(prompt_lengths):.1f} chars")
    
    # 3. Response analysis
    response_lengths = [len(ex["response"]) for ex in examples]
    print(f"\nResponse Statistics:")
    print(f"Average response length: {np.mean(response_lengths):.1f} chars")
    print(f"Response length range: {min(response_lengths)} - {max(response_lengths)}")
    
    # 4. Domain coverage
    domains = defaultdict(int)
    for ex in examples:
        domains[ex["metadata"]["domain"]] += 1
        
    plt.figure(figsize=(10, 4))
    domain_df = pd.DataFrame(list(domains.items()), columns=["Domain", "Count"])
    sns.barplot(data=domain_df, x="Domain", y="Count")
    plt.title("Distribution of Training Examples Across Domains")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze the generated examples
analyze_training_data(examples)

# Save analysis results
analysis_results = {
    "total_examples": len(examples),
    "avg_energy": float(np.mean([ex["energy"] for ex in examples])),
    "unique_domains": len(set(ex["metadata"]["domain"] for ex in examples)),
    "timestamp": pd.Timestamp.now().isoformat()
}

with open("training_data/analysis_results.json", "w") as f:
    json.dump(analysis_results, f, indent=2)

print("\nAnalysis results saved to training_data/analysis_results.json")